In [1]:
pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 13.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 17.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 13.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 15.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.3 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.2.3
    Uninstalling urllib3-2.2.3:
      Successfully uninstalled urllib3-2.2.3
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0
    Uninstalling python-dateutil-2.9.0:
      Successfully uninstalled python-dateutil-2.9.0
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.

In [2]:
pip install tabulate

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
import boto3
import glob
from urllib.parse import unquote_plus
from pyspark.sql import SparkSession
import uuid
import sys
import psycopg2
import logging
import pandas as pd
import janitor
from datetime import datetime
from pandas.api.types import infer_dtype
from pyspark.sql.functions import col, trim, regexp_replace, row_number, sha2, concat_ws, conv, lit, concat, coalesce, isnan,  isnull, regexp_replace,  monotonically_increasing_id, format_string
import tempfile
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, DoubleType, StringType, TimestampType, LongType
import numpy as np
import hashlib
from pyspark.sql.functions import udf
import warnings
from typing import Dict, Optional
from tabulate import tabulate

In [2]:
input_path = 'TEIGA_TMI_ADISSEO_CONTROL_AVANCE_SOPORTACION.csv'

In [3]:
df = pd.read_csv(input_path, header=0,sep=',', usecols=range(43))

In [20]:
print(df.columns)

Index(['NMREV', 'ID MONMEGA', 'TAG', 'AREA', 'E3DID', 'ISOID', 'CLASS',
       'DIAM ITEM', 'SUPPORTID', 'SUPPORTMARKALL', 'SUPPORTMARK or SIKLATYPE',
       'DATA1', 'DATA2or SIKLAPARTNUMBER', 'DATA3', 'DATA4', 'DATA5', 'DATA6',
       'DATA7', 'DATA8', 'DATA9', 'DATA10', 'REMARKS', 'VIGASparaCARTELAS',
       'OLD SUPPORT ID', 'FABRICANTE', 'ESTADODEFABRICACIONNUEVOFORMATO',
       'No ENTREGA NUEVO FORMATO', 'Cuenta', 'FECHA', 'MONTAJE', 'FECHA2',
       'PESOKG', 'Observación', 'CONTEO', 'Weld', 'IV', 'FechaIV', 'Result',
       'PT', 'FechaPT', 'ResultPT', 'TESTPACKASOCIADO', 'SUPPORT',
       'indexsupportid', 'installed'],
      dtype='object')


In [4]:
def clean_supportid(val):
    if pd.isnull(val):
        return None
    if val.startswith("/SPS-"):
        return val
    return val.split("_")[0]

def assign_provider(val):  # fixed spelling
    if pd.isnull(val):
        return None
    elif val != 'TEIGA-TMI':
        return 'TECHNIP'
    else:
        return val

df = df[df['E3DID'].notna()]
df = df.rename(columns={'NMREV': 'NMREVOLD'})
df['NMREV'] = df['NMREVOLD'].apply(assign_provider)
df['SUPPORTID'] = df['SUPPORTID'].apply(clean_supportid)
df['indexsupportid'] = df.groupby(['E3DID','SUPPORTID']).cumcount() + 1

select_columns = [
    'NMREV','E3DID', 'SUPPORTID', 'ESTADODEFABRICACIONNUEVOFORMATO',
    'FECHA', 'FECHA2', 'MONTAJE', 'indexsupportid'
]

In [7]:
df_light = df[select_columns].copy()
fecha2_notnull = df_light['FECHA2'].notnull() & (df_light['FECHA2'].astype(str).str.strip() != '')
fecha_notnull = df_light['FECHA'].notnull() & (df_light['FECHA'].astype(str).str.strip() != '')
df_light['is_installed_row'] = fecha2_notnull & (df_light['MONTAJE'] == 1)
df_light['is_recieved_row'] = fecha_notnull & (df_light['ESTADODEFABRICACIONNUEVOFORMATO'] == 'ENTREGA')

In [8]:
def installed_flag(group):
    if group['is_installed_row'].all():
        return 'installed'
    else:
        return 'not installed'
def recieved_flag(group):
    if group['is_recieved_row'].all():
        return 'recieved'
    else:
        return 'not recieved'

In [9]:
flags_i = df_light.groupby(['E3DID', 'SUPPORTID']).apply(installed_flag).reset_index()
flags_i.columns = ['E3DID', 'SUPPORTID', 'installed_flag']

df_end1 = df_light.merge(flags_i, on=['E3DID', 'SUPPORTID'], how='left')
df_end1 = df_end1.drop(columns=['is_installed_row'])

flags_r = df_end1.groupby(['E3DID', 'SUPPORTID']).apply(recieved_flag).reset_index()
flags_r.columns = ['E3DID', 'SUPPORTID', 'recieved_flag']

df_end2 = df_end1.merge(flags_r, on=['E3DID', 'SUPPORTID'], how='left')
df_end2 = df_end2.drop(columns=['is_recieved_row'])

/tmp/ipykernel_11837/3630678420.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  flags_i = df_light.groupby(['E3DID', 'SUPPORTID']).apply(installed_flag).reset_index()
/tmp/ipykernel_11837/3630678420.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  flags_r = df_end1.groupby(['E3DID', 'SUPPORTID']).apply(recieved_flag).reset_index()


In [10]:
df_end2.to_csv('output.csv',sep=',', index=False)

In [9]:
print(tabulate(df_end.head(100), headers='keys',tablefmt='psql'))

+----+--------------------------------------+-----------------------------+-----------------------------------+---------------------+---------------------+-----------+------------------+------------------+
|    | E3DID                                | SUPPORTID                   | ESTADODEFABRICACIONNUEVOFORMATO   | FECHA               | FECHA2              |   MONTAJE |   indexsupportid | installed_flag   |
|----+--------------------------------------+-----------------------------+-----------------------------------+---------------------+---------------------+-----------+------------------+------------------|
|  0 | /A10001-100-AGL-10395-U1BC2A-CF_01   | /800228-A10001-MP-STD-00559 | ENTREGA                           | 2024-11-11T00:00:00 | 2025-03-19T00:00:00 |         1 |                1 | installed        |
|  1 | /A10001-100-AGL-10395-U1BC2A-CF_01   | /800228-A10001-MP-STD-00559 | ENTREGA                           | 2024-11-26T00:00:00 | 2025-03-19T00:00:00 |         1 |         

In [9]:
df_light2 = df[select_columns].copy()
group_keys = ['E3DID', 'SUPPORTID']


df_light2['is_installed'] = (df_light2['FECHA2'].notna()) & (df_light2['MONTAJE'] == 1)

installed_flags = df_light2.groupby(group_keys)['is_installed'].transform('all')

df_light2['installed'] = installed_flags.map({True: 'installed', False: 'not installed'})

df_light2['is_recieved'] = (df_light2['FECHA'].notna()) & (df_light2['ESTADODEFABRICACIONNUEVOFORMATO'] == 'ENTREGA')
recieved_flags = df_light2.groupby(group_keys)['is_recieved'].transform('all')
df_light2['recieved'] = recieved_flags.map({True: 'recieved', False: 'not recieved'})

In [10]:
df_light2.to_csv('outpu2.csv', sep=',')

In [11]:
import awswrangler as wr
wr.athena.to_iceberg

In [17]:
print(tabulate(df_light2.head(100), headers='keys',tablefmt='psql'))

+----+--------------------------------------+-----------------------------+-----------------------------------+---------------------+---------------------+-----------+------------------+---------+---------------+
|    | E3DID                                | SUPPORTID                   | ESTADODEFABRICACIONNUEVOFORMATO   | FECHA               | FECHA2              |   MONTAJE |   indexsupportid | valid   | installed     |
|----+--------------------------------------+-----------------------------+-----------------------------------+---------------------+---------------------+-----------+------------------+---------+---------------|
|  0 | /A10001-100-AGL-10395-U1BC2A-CF_01   | /800228-A10001-MP-STD-00559 | ENTREGA                           | 2024-11-11T00:00:00 | 2025-03-19T00:00:00 |         1 |                1 | True    | installed     |
|  1 | /A10001-100-AGL-10395-U1BC2A-CF_01   | /800228-A10001-MP-STD-00559 | ENTREGA                           | 2024-11-26T00:00:00 | 2025-03-19T00:

In [ ]:
import csv
with open(inputh, 'r') as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        if i == 9034:  # 0-based index equivalent to line 9035
            print(f"Columns: {len(row)}, Data: {row}")

In [2]:
inputtableA = 'tableA.csv'
inputtableB = 'tableB.csv'

In [3]:
tableA = pd.read_csv(inputtableA, header=0, sep=',',low_memory=False)
tableB = pd.read_csv(inputtableB, header=0, sep=',')

In [4]:
count_a = tableA.groupby('e3did')['record'].count().reset_index(name='count_a')
count_b = tableB.groupby('e3did')['record'].count().reset_index(name='count_b')

In [5]:
max_counts = pd.merge(count_a, count_b, on='e3did', how='outer').fillna(0)
max_counts['max_count'] = max_counts[['count_a', 'count_b']].max(axis=1)

In [7]:
expanded = []
for _, row in max_counts.iterrows():
    expanded.extend([(row['e3did'], i+1) for i in range(int(row['max_count']))])

In [8]:
expanded_df = pd.DataFrame(expanded, columns=['e3did', 'record'])

In [9]:
result= (
    expanded_df
    .merge(tableA, on=['e3did', 'record'], how='left')
    .merge(tableB, on=['e3did', 'record'], how='left')
)

In [10]:
final_df = result.sort_values(['e3did', 'record']).reset_index(drop=True)

In [19]:
print(final_df.columns)

Index(['e3did', 'record', 'total', 'date_execut', 'supportid', 'montaje',
       'fecha2'],
      dtype='object')


In [17]:
print(tabulate(final_df, headers='keys', tablefmt='psql'))

+----+-------------------------------------+----------+---------+-----------------+-----------------------------+-----------+----------------+
|    | e3did                               |   record |   total | date_execut     | supportid                   |   montaje | fecha2         |
|----+-------------------------------------+----------+---------+-----------------+-----------------------------+-----------+----------------|
|  0 | /A10001-100-AGL-10395-U1BC2A-CF_01  |        1 |    1    | 8/19/2024 0:00  | /800228-A10001-MP-STD-00559 |         1 | 3/19/2025 0:00 |
|  1 | /A10001-100-AGL-10395-U1BC2A-CF_01  |        2 |    1    | 8/19/2024 0:00  | /800228-A10001-MP-STD-00559 |         1 | 3/19/2025 0:00 |
|  2 | /A10001-100-AGL-10395-U1BC2A-CF_01  |        3 |    1    | 8/19/2024 0:00  | /800228-A10001-MP-STD-01306 |       nan | nan            |
|  3 | /A10001-100-AGL-10395-U1BC2A-CF_01  |        4 |    0.75 | 8/20/2024 0:00  | /800228-A10001-MP-STD-00571 |       nan | nan            |